This section:

• Imports required libraries  
• Configures Brightway2  
• Connects to ecoinvent  
• Ensures the required LCA database is available  

🔎 USER-DEFINED PARAMETERS are clearly marked below.

In [22]:
# ============================
# SYSTEM IMPORTS (not user-defined)
# ============================

import os
import pandas as pd
import numpy as np
import rasterio
import matplotlib.pyplot as plt

import bw2io as bi
import bw2data as bd
from bw2calc import LCA

In [24]:
# ============================
# USER-DEFINED SETTINGS
# ============================

ECOINVENT_VERSION = "3.10"
SYSTEM_MODEL = "cutoff"      # Options: cutoff / apos / consequential / EN15804
ECOINVENT_USERNAME = os.getenv("ECOINVENT_USERNAME")
ECOINVENT_PASSWORD = os.getenv("ECOINVENT_PASSWORD")


For security, set your ecoinvent credentials as environment variables:

On Mac/Linux:
export ECOINVENT_USERNAME="your_username"
export ECOINVENT_PASSWORD="your_password"

On Windows (PowerShell):
setx ECOINVENT_USERNAME "your_username"
setx ECOINVENT_PASSWORD "your_password"

This prevents passwords from being hard-coded into the notebook.

In [27]:
# -----------------------------------------
# ECOINVENT CONFIGURATION
# -----------------------------------------

db_name = 'ecoinvent-3.10-cutoff'

if db_name not in bd.databases:

    print(f"Importing {db_name}...")

    bi.import_ecoinvent_release(
        version='3.10',
        system_model='cutoff',  # cutoff / apos / consequential / EN15804
        username=ECOINVENT_USERNAME,
        password=ECOINVENT_PASSWORD
    )

else:
    print("{db_name} is already present in the project")


db = bd.Database(db_name)


{db_name} is already present in the project


This section defines the physical and chemical properties of crop residues used in the biochar model, as well as other key constants and models necessary to calculate both biochar sequestration potential for different crop residues and upstream emissions for each crop residue type.

This section includes first user-defined parameters and constants with default values inputted, and afterward literature-derived values and formulas.


In [30]:
# ============================
# USER-DEFINED PARAMETERS
# ============================

PYROLYSIS_TEMPERATURE_C = 550     # degrees Celsius
MOLECULAR_CONVERSION = 44 / 12    # C → CO2 conversion factor

# Biochar yield equation parameters (Karan et al., 2023)
# Biochar C yield fraction:
# Y = 0.126 + 0.273 * Lignin + 0.539 * exp(-0.004 * T)

# ============================
# USER-DEFINED SCENARIO INPUTS
# ============================

LCA_METHOD = ('ecoinvent-3.10', 'IPCC 2021', 'climate change', 'global warming potential (GWP100)')

TRANSPORT_DISTANCE_KM = 15  # one-way distance, based on midwestern US corn density and a biochar plant processing 10 tons/hr. https://www.css.cornell.edu/faculty/lehmann/publ/ES&T%2044,%20827-833,%202010%20supporting%20online%20Roberts.pdf
TRUCK_EMISSION_FACTOR = 0.00000545  # kg CO2e per kg-km (dry basis) | computed using heavy duty diesel truck info from canada. desjardins et al. 2024, https://link.springer.com/article/10.1007/s42773-024-00352-z/tables/3

ROUND_TRIP = True


In [32]:
# ============================
# LITERATURE-DERIVED CONSTANTS
# ============================

# H/C atomic ratio assumption, from rice at 500C (currently uniform across crops)
# Source: OAB supplement (rice value applied to all crops for now)
h2c = {"rice production, non-basmati": 0.359, "maize grain production": 0.359, "wheat grain production": 0.359, "soybean production": 0.359, "barley grain production": 0.359, "millet production": 0.359, "rape seed production": 0.359, "sugarcane production": 0.359}

# Main product and some residue yields are from ecoinvent. Yields in function form are from Karan et al. 2023 paper supplement
yields = {"rice production, non-basmati": (6639.3, 3319.5), "maize grain production": (5800, 5800 * (1000*(-0.1807*np.log(0.001)+1.3796))), "wheat grain production": (2725, 691), "soybean production": (2710, 2710 * 3869*np.exp(-0.178*0.001)), "barley grain production": (3100, 3100 * 1822*np.exp(-0.149*0.001)), "millet production": (1010, 1010 * (-0.95*0.001 + 4.38)*1000), "rape seed production": (2110, 2110 * 3028*np.exp(-0.2 * 0.001)), "sugarcane production": (66300, 66300 * 1000*(1.328*np.exp(-0.06*0.001)))}

# Harvest fractions taken from Karan et al. 2023 supplement
harv_frac = {"rice production, non-basmati": 0.75, "maize grain production": 0.5, "wheat grain production": 0.4, "soybean production": 0.4, "barley grain production": 0.4, "millet production": 0.4, "rape seed production": 0.5, "sugarcane production": 0.75}

# Helper titles for bridging datasets
crops = {"rice production, non-basmati": "rice", "maize grain production": "maiz", "wheat grain production": "whea", "soybean production": "soyb", "barley grain production": "barl", "millet production": "pmil", "rape seed production": "rape", "sugarcane production": "sugc"}
# Dictionary of {activity title in ecoinvent: [residue feedstock, main product]}
activity_name_target_list = {"rice production, non-basmati": ["straw", "Rice"], "maize grain production": ["straw", "Maize"], "wheat grain production": ["straw", "Wheat"], "soybean production": ["straw", "Soybean"], "barley grain production": ["straw", "Barley"], "millet production": ["Straw", "Barley"], "rape seed production": ["Straw", "Rapeseed"], "sugarcane production": ["Bagasse", "Sugarcane"]}


# BC100 stability model (From OAB supplement)
def calculate_bc100(h2c_ratio):
    return 1.22 - 0.6 * h2c_ratio


# From Karan et al. 2023 supplement
def load_crop_residue_data(csv_path):
    required_columns = [
        "Dry weight conversion factor",
        "Carbon fraction (% DAF)",
        "Carbon fraction (% DM)",
        "Lignin content (% DM)",
        "LHV (MJ/kg DAF)",
    ]

    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Crop residue data file not found: {csv_path}")

    crop_residue_df = pd.read_csv(csv_path)
    crop_residue_data = {}

    for row in crop_residue_df.to_dict(orient="records"):
        crop = row.pop("Crop")
        residue = row.pop("Residue")
        crop_residue_data.setdefault(crop, {})[residue] = {
            column: float(row[column]) for column in required_columns
        }

    return crop_residue_data


crop_residue_data = load_crop_residue_data("crop_residue_data.csv")


In [34]:
# Compute transportation emissions

if ROUND_TRIP:
    TRANSPORT_EMISSIONS = TRUCK_EMISSION_FACTOR * TRANSPORT_DISTANCE_KM * 2
else:
    TRANSPORT_EMISSIONS = TRUCK_EMISSION_FACTOR * TRANSPORT_DISTANCE_KM


This section:

1️⃣ Fetches crop production activities from ecoinvent  
2️⃣ Computes GWP100 climate impacts  
3️⃣ Allocates impacts between crop + residue  
4️⃣ Applies carbon-based reallocation  
5️⃣ Adds transport emissions  
6️⃣ Applies biochar stability correction  
7️⃣ Computes net CO₂ sequestration per kg residue  

Output:
Net emissions after biochar production [kg CO₂e/kg residue]


In [37]:
# CONFIGURATION

# -----------------------------------------
# Helper to normalize keys and convert to float
# -----------------------------------------
def normalize_key(name):
    n = name.lower().replace("-", " ").replace("  ", " ").strip()
    mapping = {
        "functional unit": "Functional unit",
        "allocation factor": "Allocation factor",
        "carbon allocation": "Carbon allocation [kg]",
        "carbon content": "Carbon content",
        "carbon content, fossil": "Carbon content, fossil",
        "carbon content, non fossil": "Carbon content, non-fossil",
        "dry mass": "Dry mass [kg]",
        "heating value, net": "Heating value, net [MJ]",
        "price": "Price [EUR2005]",
        "true value": "True value",
        "water content": "Water content",
        "water in wet mass": "Water in wet mass [kg]",
        "wet mass": "Wet mass [kg]",
        "yield": "Yield",
        "climate change impact": "Climate change impact [kg CO2e/f.u.]",
        "unit production ratio (relative to char input)": "Unit production ratio (relative to char input)",
        "climate change impact [kg co2e/bundle]": "climate change impact [kg CO2e/bundle]",
        "effective carbon allocation": "Effective carbon allocation",
        "effective climate change impact  [kg co2e/bundle]": "Effective climate change impact  [kg CO2e/bundle]",
        "effective climate change impact  [kg co2e/kg feedstock]": "Effective climate change impact  [kg CO2e/kg feedstock]",
        "biomass capture  [kg co2e/kg feedstock]": "Biomass capture  [kg CO2e/kg feedstock]",
        "net emissions after biochar prod [kg co2e/kg residue]": "Net emissions after biochar prod [kg CO2e/kg residue]"
    }
    return mapping.get(n, name)

def to_float(value):
    if value is None or value == "":
        return 0.0
    if isinstance(value, (int, float)):
        return float(value)
    # Remove units and whitespace
    try:
        return float(str(value).split()[0])
    except:
        return 0.0

fields = [
    "Functional unit",
    "Allocation factor",
    "Carbon allocation [kg]",
    "Carbon content",
    "Carbon content, fossil",
    "Carbon content, non-fossil",
    "Dry mass [kg]",
    "Heating value, net [MJ]",
    "Price [EUR2005]",
    "True value",
    "Water content",
    "Water in wet mass [kg]",
    "Wet mass [kg]",
    "Yield",
    "Climate change impact [kg CO2e/f.u.]",
    "Unit production ratio (relative to char input)",
    "Climate change impact [kg CO2e/bundle]",
    "Effective carbon allocation",
    "Effective climate change impact [kg CO2e/bundle]",
    "Effective climate change impact  [kg CO2e/kg feedstock]",
    "Biomass capture  [kg CO2e/kg feedstock]",
    "Net emissions after biochar prod [kg CO2e/kg residue]"
]

In [ ]:
# Fetching properties and computing upstream emissions, net emissions for each type of crop in each location

data_by_act = {}

# for each crop
for key, val in activity_name_target_list.items():
    activity_name_target = key # production name
    char_stock = val[0] # residue stock name
    col_name = val[1] # coloquial name (e.g. rice)
    
    method_name = ('ecoinvent-3.10', 'IPCC 2021', 'climate change', 'global warming potential (GWP100)') 

    
    # -----------------------------------------
    # 3. Find activities
    # -----------------------------------------

    # for each product (residue feedstock, main product)
    activities = [act for act in db if act['name'] == activity_name_target]
    if not activities:
        raise ValueError(f"Activity '{activity_name_target}' not found in database.")
    
    print(f"Found {len(activities)} activities for '{activity_name_target}'")
    
    # -----------------------------------------
    # 4. Group activities by location
    # -----------------------------------------
    activities_by_location = {}
    for act in activities:
        loc = act['location']
        activities_by_location.setdefault(loc, []).append(act)
    
    data_by_location = {}
    
    # -----------------------------------------
    # 5. Fetch data for this particular crop in a given location
    # -----------------------------------------
    
    
    for loc, acts in activities_by_location.items():
        
        carbon_dict = {}
        dry_mass_dict = {}
        allocation_dict = {}
        
        # Step 1: Populate data and extract land, allocation, carbon, dry mass
        for act in acts:
            
            ref_exchanges = [exc for exc in act.exchanges() if exc['type'] == 'production']
            exc = ref_exchanges[0]
            ref_name = exc['name'] # where ref_name is straw, rice, etc (name of the reference product)
    
            data_by_location.setdefault(loc, {})[ref_name] = {}
            data_by_location[loc][ref_name]["Functional unit"] = f"{exc['amount']} {exc['unit']}"

            alloc = 1.0
            for name, prop in exc.get('properties', {}).items():
                if 'allocation' in name.lower():
                    alloc = prop.get('amount', 1.0) if isinstance(prop, dict) else prop
            allocation_dict[ref_name] = alloc
    
            # Properties
            for name, prop in exc.get('properties', {}).items():
                key = normalize_key(name)
                if isinstance(prop, dict):
                    amount = prop.get('amount')
                    unit = prop.get('unit', "")
                    val = f"{amount} {unit}".strip()
                    data_by_location[loc][ref_name][key] = val
                else:
                    data_by_location[loc][ref_name][normalize_key(name)] = prop
    
            # Other details
            for k, v in exc.get('otherDetails', {}).items():
                key = normalize_key(k)
                data_by_location[loc][ref_name][key] = v
        
    
            # Carbon allocation & dry mass
            carbon_dict[ref_name] = to_float(data_by_location[loc][ref_name].get("Carbon allocation [kg]", 0))
            dry_mass_dict[ref_name] = to_float(data_by_location[loc][ref_name].get("Dry mass [kg]", 1.0))
    
            # -----------------------------------------
            # Compute climate change impact per functional unit
            # -----------------------------------------
            demand = {act: exc['amount']}
            lca = LCA(demand, method_name)
            lca.lci()
            lca.lcia()
            cc_score = lca.score
            data_by_location[loc][ref_name]["Climate change impact [kg CO2e/f.u.]"] = cc_score

        if len(acts) == 1: # if there is only one activity in ecoinvent
            main = ref_name
            data_by_location.setdefault(loc, {})[char_stock] = {}
            data_by_location[loc][char_stock]["Functional unit"] = "1 kg" 
            total_cc_score = data_by_location[loc][main]["Climate change impact [kg CO2e/f.u.]"]
            total_yield = yields[activity_name_target][0] + yields[activity_name_target][1]
            # allocating by mass
            data_by_location[loc][main]["Climate change impact [kg CO2e/f.u.]"] = yields[activity_name_target][0]/total_yield * total_cc_score
            data_by_location[loc][char_stock]["Climate change impact [kg CO2e/f.u.]"] = yields[activity_name_target][1]/total_yield * total_cc_score

            dry_mass_dict[char_stock] = crop_residue_data[activity_name_target_list[activity_name_target][1]][activity_name_target_list[activity_name_target][0].capitalize()]["Dry weight conversion factor"]
            carbon_dict[char_stock] = dry_mass_dict[char_stock] * 0.01 * crop_residue_data[activity_name_target_list[activity_name_target][1]][activity_name_target_list[activity_name_target][0].capitalize()]["Carbon fraction (% DM)"]
            
        # how much kg of carbon per 1kg of dry feedstock mass 
            


        
# which fields to populate to calc. effective climate change impact (co2e/feedstock)
        
        # -----------------------------------------
        # Fetch yields, upr from earlier dataset
        # -----------------------------------------
    
        main_product = max(allocation_dict, key=lambda x: allocation_dict[x])
    
        for ref_name in data_by_location[loc].keys():
            ind = 1
            if ref_name == main_product:
                ind = 0
            data_by_location[loc][ref_name]["Yield"] = yields[activity_name_target][ind]
    
        for ref_name in data_by_location[loc].keys():
            yield_val = to_float(data_by_location[loc][ref_name].get("Yield", 0))
            char_yield = to_float(data_by_location[loc].get(char_stock, {}).get("Yield", 1))
            upr = yield_val / char_yield # Unit Production Ratio to residue production
            data_by_location[loc][ref_name]["Unit production ratio (relative to char input)"] = upr


        # -----------------------------------------
        # Climate change impact per bundle
        # -----------------------------------------
        for ref_name in data_by_location[loc].keys():
            cc_fu = to_float(data_by_location[loc][ref_name].get("Climate change impact [kg CO2e/f.u.]", 0))
            upr = to_float(data_by_location[loc][ref_name].get("Unit production ratio (relative to char input)", 0)) 
            data_by_location[loc][ref_name]["Climate change impact [kg CO2e/bundle]"] = cc_fu * upr
    
        # -----------------------------------------
        # Effective carbon allocation & effective CC impact
        # -----------------------------------------
        weighted_sum = sum(carbon_dict[r] * to_float(data_by_location[loc][r]["Unit production ratio (relative to char input)"]) for r in data_by_location[loc].keys())
        
        for ref_name in data_by_location[loc].keys():
            upr = to_float(data_by_location[loc][ref_name]["Unit production ratio (relative to char input)"])
            data_by_location[loc][ref_name]["Effective carbon allocation"] = carbon_dict[ref_name] * upr / weighted_sum


        # REALLOCATION (using lexi's formulas)
        total_cc_bundle = sum(to_float(data_by_location[loc][r]["Climate change impact [kg CO2e/bundle]"]) for r in data_by_location[loc].keys())
        for ref_name in data_by_location[loc].keys():
            upr = to_float(data_by_location[loc][ref_name]["Unit production ratio (relative to char input)"])
            eff_carbon = (carbon_dict[ref_name] * upr / weighted_sum) 

            data_by_location[loc][ref_name]["Effective climate change impact [kg CO2e/bundle]"] = eff_carbon * total_cc_bundle 
                
            # Effective climate per kg feedstock
            dry_mass = dry_mass_dict[ref_name] if dry_mass_dict[ref_name] else 1
            data_by_location[loc][ref_name]["Effective climate change impact  [kg CO2e/kg feedstock]"] = \
                data_by_location[loc][ref_name]["Effective climate change impact [kg CO2e/bundle]"] / dry_mass
    
            # Biomass capture
            data_by_location[loc][ref_name]["Biomass capture  [kg CO2e/kg feedstock]"] = \
                carbon_dict[ref_name] * MOLECULAR_CONVERSION * upr / dry_mass


            # now it's all on a dry basis! Populate fields for ref product at given location
            if ref_name == char_stock:
                residue_properties = crop_residue_data[activity_name_target_list[activity_name_target][1]][activity_name_target_list[activity_name_target][0].capitalize()]
                biochar_crop_key = activity_name_target_list[activity_name_target][1]
                biochar_yield = 0.126 + 0.273 * residue_properties["Lignin content (% DM)"] + 0.539 * np.exp(-0.004 * PYROLYSIS_TEMPERATURE_C)
                biochar_carbon_content = residue_properties["Carbon fraction (% DAF)"] / residue_properties["Carbon fraction (% DM)"]
                biochar_cdr_per_biochar = MOLECULAR_CONVERSION * biochar_carbon_content
                h2c_at_500c = h2c[activity_name_target]
                data_by_location[loc][ref_name]["Implied BC100"] = calculate_bc100(h2c_at_500c)
                data_by_location[loc][ref_name]["End of Life loss [kg CO2/kg feedstock]"] = (carbon_dict[ref_name] * (44/12) / dry_mass_dict[ref_name]) - min(1, data_by_location[loc][ref_name]["Implied BC100"]) * carbon_dict[ref_name] * 44/12 / dry_mass_dict[ref_name]  
                data_by_location[loc][ref_name]["Biochar carbon content [kg/kg]"] = biochar_carbon_content
                data_by_location[loc][ref_name]["Biochar CDR [kg CO2/kg biochar]"] = biochar_cdr_per_biochar
                data_by_location[loc][ref_name]["Biochar CDR per feedstock"] = biochar_cdr_per_biochar * biochar_yield
                data_by_location[loc][ref_name]["Net emissions after biochar prod [kg CO2e/kg residue]"] = data_by_location[loc][ref_name]["Effective climate change impact  [kg CO2e/kg feedstock]"] + TRANSPORT_EMISSIONS + data_by_location[loc][ref_name]["End of Life loss [kg CO2/kg feedstock]"] - data_by_location[loc][ref_name]["Biochar CDR per feedstock"]
                pyrolysis_loss = data_by_location[loc][ref_name]["Biomass capture  [kg CO2e/kg feedstock]"] - data_by_location[loc][ref_name]["Biochar CDR per feedstock"]
                requested_print_sections = [
                    ("Feedstock properties", [
                        ("Functional unit", data_by_location[loc][ref_name].get("Functional unit")),
                        ("Allocation factor", data_by_location[loc][ref_name].get("Allocation factor", allocation_dict.get(ref_name))),
                        ("Carbon allocation [kg]", data_by_location[loc][ref_name].get("Carbon allocation [kg]", carbon_dict.get(ref_name))),
                        ("Carbon content", data_by_location[loc][ref_name].get("Carbon content")),
                        ("Carbon content, fossil", data_by_location[loc][ref_name].get("Carbon content, fossil")),
                        ("Carbon content, non-fossil", data_by_location[loc][ref_name].get("Carbon content, non-fossil")),
                        ("Dry mass [kg]", data_by_location[loc][ref_name].get("Dry mass [kg]", dry_mass_dict.get(ref_name))),
                        ("Heating value, net [MJ]", data_by_location[loc][ref_name].get("Heating value, net [MJ]")),
                        ("Price [EUR2005]", data_by_location[loc][ref_name].get("Price [EUR2005]")),
                        ("True value", data_by_location[loc][ref_name].get("True value")),
                        ("Water content", data_by_location[loc][ref_name].get("Water content")),
                        ("Water in wet mass [kg]", data_by_location[loc][ref_name].get("Water in wet mass [kg]")),
                        ("Wet mass [kg]", data_by_location[loc][ref_name].get("Wet mass [kg]")),
                        ("Yield, wet mass [kg/ha]", data_by_location[loc][ref_name].get("Yield, wet mass [kg/ha]", data_by_location[loc][ref_name].get("Yield"))),
                        ("Climate change impact [kg CO2e/f.u.]", data_by_location[loc][ref_name].get("Climate change impact [kg CO2e/f.u.]")),
                        ("Unit production ratio (relative to char input)", data_by_location[loc][ref_name].get("Unit production ratio (relative to char input)")),
                        ("Climate change impact [kg CO2e/bundle]", data_by_location[loc][ref_name].get("Climate change impact [kg CO2e/bundle]")),
                    ]),
                    ("Carbon allocating", [
                        ("Effective carbon allocation", data_by_location[loc][ref_name].get("Effective carbon allocation")),
                        ("Climate change impact [kg CO2e/bundle]", data_by_location[loc][ref_name].get("Effective climate change impact [kg CO2e/bundle]")),
                        ("Climate change impact [kg CO2e/kg feedstock]", data_by_location[loc][ref_name].get("Effective climate change impact  [kg CO2e/kg feedstock]")),
                        ("Biomass capture [kg CO2e/kg feedstock]", data_by_location[loc][ref_name].get("Biomass capture  [kg CO2e/kg feedstock]")),
                    ]),
                    ("Biochar / pyrolysis", [
                        ("H/C at 500C", h2c_at_500c),
                        ("Implied BC100", data_by_location[loc][ref_name].get("Implied BC100")),
                        ("Biochar carbon content [kg/kg]", data_by_location[loc][ref_name].get("Biochar carbon content [kg/kg]")),
                        ("Biochar CDR [kg CO2/kg biochar]", data_by_location[loc][ref_name].get("Biochar CDR [kg CO2/kg biochar]")),
                        ("Lignin content [kg lignin/kg feedstock]", residue_properties["Lignin content (% DM)"] * 0.01),
                        ("Biochar mass yield [kg biochar/kg feedstock]", None),
                        ("Biochar CDR per feedstock", data_by_location[loc][ref_name].get("Biochar CDR per feedstock")),
                        ("Pyrolysis loss [kg CO2/kg feedstock]", pyrolysis_loss),
                        ("End of life loss [kg CO2/kg feedstock]", data_by_location[loc][ref_name].get("End of Life loss [kg CO2/kg feedstock]")),
                        ("Pyrolysis temperature [deg C]", PYROLYSIS_TEMPERATURE_C),
                    ]),
                ]
                print("activity name: " + str(activity_name_target))
                print("Location: " + str(loc))
                print("Reference product: " + ref_name)
                for section_name, section_fields in requested_print_sections:
                    print(section_name)
                    for label, value in section_fields:
                        if value is None:
                            print(f"{label}: no corresponding variable currently defined")
                        else:
                            print(f"{label}: {value}")
                    print()
                print("Net emissions after biochar prod [kg CO2e/kg residue]: " + str(data_by_location[loc][ref_name]["Net emissions after biochar prod [kg CO2e/kg residue]"]) + "\n")

            else:
                data_by_location[loc][ref_name]["Net emissions after biochar prod [kg CO2e/kg residue]"] = None

    data_by_act[activity_name_target] = data_by_location
    


Found 6 activities for 'rice production, non-basmati'
activity name: rice production, non-basmati
Location: US
Reference product: straw
Feedstock properties
Functional unit: 1 kg
Allocation factor: no corresponding variable currently defined
Carbon allocation [kg]: 0.3698100000000001
Carbon content: no corresponding variable currently defined
Carbon content, fossil: no corresponding variable currently defined
Carbon content, non-fossil: no corresponding variable currently defined
Dry mass [kg]: 0.9
Heating value, net [MJ]: no corresponding variable currently defined
Price [EUR2005]: no corresponding variable currently defined
True value: no corresponding variable currently defined
Water content: no corresponding variable currently defined
Water in wet mass [kg]: no corresponding variable currently defined
Wet mass [kg]: no corresponding variable currently defined
Yield, wet mass [kg/ha]: 3319.5
Climate change impact [kg CO2e/f.u.]: 0.5971535224752189
Unit production ratio (relative to 

The below cell has the user enter a folder path to the folder containing res_prod files from the Karan et al. 2023 supplement.

Then, waterfall plots and raster files are produced to show net emissions calculations for each crop and net biochar-CO2e emissions raster files covering the global geography.

In [ ]:
# PLOTS
# ----------------------------------
# User inputs
# ----------------------------------
folder_path = r"C:\Users\Maanit\Downloads\dataverse_files (5)" # folder path of residue files. downloadable from: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/TSU0IE

# ----------------------------------
# Loop over crops
# ----------------------------------
total_emissions_all = 0

total_biochar_raster = None
extent_all = None

for crop, n in crops.items():

    res_prod_filename = f"res_prod_{n}.tif"
    res_prod_path = os.path.join(folder_path, res_prod_filename)
    char_stock = activity_name_target_list[crop][0]

    if not os.path.exists(res_prod_path):
        print(f"⚠️ File not found: {res_prod_filename}")
        continue


    # Waterfall plot

    # Terms and values
    terms = {
        "Effective climate change impact [kg CO2e/kg feedstock]": data_by_location[loc][ref_name]["Effective climate change impact  [kg CO2e/kg feedstock]"],
        "Transportation Emissions [kg CO2e/kg feedstock]": TRANSPORT_EMISSIONS,
        "End of Life loss [kg CO2/kg feedstock]": data_by_location[loc][ref_name]["End of Life loss [kg CO2/kg feedstock]"],
        "Biochar CDR per feedstock [kg CO2e/kg feedstock]": -data_by_location[loc][ref_name]["Biochar CDR per feedstock"]
    }
    
    vals = list(terms.values())
    cum = [0]+[sum(vals[:i+1]) for i in range(len(vals))]
    colors = ["red" if v>0 else "green" for v in vals]
    plt.title(f"Emissions Components: {crop}", fontsize=14)
    plt.bar(terms.keys(), vals, bottom=cum[:-1], color=colors)
    for i, (v, c) in enumerate(zip(vals, cum[:-1])):
        plt.text(i, c+v/2, f"{v:.2f}", ha='center', va='center', color='white', fontweight='bold')
    
    plt.bar("Net emissions", sum(vals), color='blue')
    plt.text(len(vals), sum(vals)/2, f"{sum(vals):.2f}", ha='center', va='center', color='white', fontweight='bold')
    
    plt.ylabel("kg CO2e/kg feedstock"); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

    
    # Read raster
    with rasterio.open(res_prod_path) as src:
        res_prod = src.read(1).astype(float)
        extent = [
            src.bounds.left, src.bounds.right,
            src.bounds.bottom, src.bounds.top
        ]

        if src.nodata is not None:
            res_prod[res_prod == src.nodata] = np.nan



    # ------------------------------
    # Plot original raster
    # ------------------------------
    plt.figure(figsize=(10, 5))
    plt.imshow(res_prod, cmap="viridis", extent=extent)
    plt.title(f"Residue Production: {crop}", fontsize=14)
    plt.colorbar(label="Residue production (Mg of dry mass/pixel) / yr") # pixel resolution: 5x5 minutes of Arc
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()

    # ------------------------------
    # Compute constrained biochar raster
    # ------------------------------

    biochar_constrained = (
        data_by_act[crop]["RoW"][char_stock]
    ["Net emissions after biochar prod [kg CO2e/kg residue]"]
        * harv_frac[activity_name_target]
        * res_prod 
    )

    # ------------------------------
    # Plot derived raster
    # ------------------------------
    plt.figure(figsize=(10, 5))
    plt.imshow(biochar_constrained, cmap="magma", extent=extent)
    plt.title(
        f"Biochar-CO2e (constrained) WITH upstream emissions: {crop}",
        fontsize=14
    )
    plt.colorbar(label="Net emitted: Mg CO₂e / yr")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()


    # ------------------------------
    # Compute biochar raster NO UPSTREAM
    # ------------------------------

    biochar_constrained_no_up = (
        data_by_act[crop]["RoW"][char_stock]
    ["Biochar CDR per feedstock"]
        * harv_frac[activity_name_target]
        * res_prod 
    )

    # ------------------------------
    # Plot derived raster
    # ------------------------------
    plt.figure(figsize=(10, 5))
    plt.imshow(biochar_constrained_no_up, cmap="magma", extent=extent)
    plt.title(
        f"Biochar (constrained) without upstream emissions: {crop}",
        fontsize=14
    )
    plt.colorbar(label="Net emitted: Mg CO₂e / yr")
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.tight_layout()
    plt.show()

        # Initialize accumulator on first iteration
    if total_biochar_raster is None:
        total_biochar_raster = np.zeros_like(biochar_constrained)
        extent_all = extent  # reuse from first raster

    
    # Add this crop to total
    total_biochar_raster -= np.nan_to_num(biochar_constrained, nan=0.0)

    # ------------------------------
    # Sum pixels
    # ------------------------------
    total_emissions = np.nansum(biochar_constrained_no_up)
    total_net_emissions = np.nansum(biochar_constrained)
    total_emissions_all = total_emissions_all + total_net_emissions

    print(
        f"Total biochar (constrained) emissions sequestered "
        f"({crop}): {total_emissions:,.3e} kg CO₂e / yr\n"
    )

    print(
        f"Total NET biochar (constrained) emissions sequestered "
        f"({crop}): {total_net_emissions:,.3e} kg CO₂e / yr\n"
    )



            


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize

# Mask exact zeros → white
masked = np.ma.masked_where(total_biochar_raster == 0, total_biochar_raster)

# Single colormap (all positive → Blues)
cmap = plt.cm.Blues
cmap.set_bad("white")  # masked zeros

# Normalize across the data range
norm = Normalize(vmin=masked.min(), vmax=masked.max())

plt.figure(figsize=(10, 5))
plt.imshow(
    masked,
    cmap=cmap,
    norm=norm,
    extent=extent_all
)
plt.title("Biochar constrained with upstream emissions", fontsize=14)
plt.colorbar(label="Mg CO₂e / yr")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()


print(
    f"Total biochar sequestration (constrained), calculated: {-1 * total_emissions_all:,.3e} Mg CO₂e / yr\n"
)



bc_path = r"C:\Users\Maanit\Downloads\dataverse_files (4)"
# ORIGINAL FROM PAPER:

bc_const_filename = "bc_prod_constrained.tif"
res_prod_path = os.path.join(bc_path, bc_const_filename)

if not os.path.exists(res_prod_path):
    print(f"⚠️ File not found: {bc_const_filename}")
    

# Read raster
with rasterio.open(res_prod_path) as src:
    bc_paper = src.read(1).astype(float) * (44/12)
    extent = [
        src.bounds.left, src.bounds.right,
        src.bounds.bottom, src.bounds.top
    ]

    if src.nodata is not None:
        bc_paper[bc_paper == src.nodata] = np.nan

# ------------------------------
# Plot original raster
# ------------------------------
plt.figure(figsize=(10, 5))
plt.imshow(bc_paper, cmap="viridis", extent=extent)
plt.title(f"BC constrained, from the paper", fontsize=14)
plt.colorbar(label="Biochar (constrained), Mg CO2e/pixel/yr")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()

paper_emissions_all = np.nansum(bc_paper)
print(
    f"Total biochar sequestration (constrained), paper: {paper_emissions_all:,.3e} Mg CO₂e / yr\n"
)

# PLOT THE DIFFERENCE

# ------------------------------
# Plot difference (1 − 2)
# ------------------------------
difference_raster = total_biochar_raster - bc_paper

plt.figure(figsize=(10, 5))
plt.imshow(difference_raster, cmap="coolwarm", extent=extent)
plt.title(
    "Difference in biochar sequestration (Including upstream − excluding upstream)",
    fontsize=14
)
plt.colorbar(label="Mg CO2e/yr")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.tight_layout()
plt.show()



In [ ]:
# STORING THE OUTPUT IN EXCEL SHEETS

for key, val in activity_name_target_list.items():
# -----------------------------------------
    # 8. Build DataFrame per location with sum column
    # -----------------------------------------
    for loc, refs in data_by_location.items():
        print(f"\n=== Location: {loc} ===")
        df = pd.DataFrame(index=fields, columns=list(refs.keys()) + ["sum"])
        for ref_name, props in refs.items():
            for field in fields:
                df.at[field, ref_name] = props.get(field, "")
        
        # Sum column
        for field in fields:
            values = []
            for ref_name in refs.keys():
                val = to_float(df.at[field, ref_name])
                values.append(val)
            df.at[field, "sum"] = sum(values)
        
        print(df)

    
    # -----------------------------------------
    # 9. Export all locations to a single Excel file
    # -----------------------------------------
    excel_filename = f"Downloads/files2026/{activity_name_target} carbon allocations, 3.10.xlsx"
    
    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        for loc, refs in data_by_location.items():
            # Create DataFrame
            df = pd.DataFrame(index=fields, columns=[col.capitalize() for col in refs.keys()] + ["Sum"])
            
            for ref_name, props in refs.items():
                col_name = ref_name.capitalize()
                for field in fields:
                    val = props.get(field, "")
                    
                    # Replace 'dimensionless' with empty string
                    if isinstance(val, str) and val.lower() == "dimensionless":
                        val = ""
                    
                    # Replace 'kilogram' with 'kg' in Functional Unit row
                    if field == "Functional unit" and isinstance(val, str):
                        val = val.replace("kilogram", "kg")
                    
                    # Remove units from all other rows except Functional unit
                    if field != "Functional unit" and isinstance(val, str):
                        val = val.split()[0] if val else val
                    
                    df.at[field, col_name] = val
            
            # Rename Yield row
            yield_row_label = f"Yield [{df.at['Functional unit', list(df.columns)[0]]}/ha]"
            df = df.rename(index={"Yield": yield_row_label})
            
            # Compute sum column numerically
            for field in df.index:
                values = []
                for col in df.columns[:-1]:  # exclude Sum column
                    val = df.at[field, col]
                    try:
                        val_float = float(val)
                    except:
                        val_float = 0.0
                    values.append(val_float)
                df.at[field, "Sum"] = sum(values)
            
            # Set top-left cell (A1) to 'Component'
            df.index.name = "Component"
            
            # Sheet name
            sheet_name = f"{activity_name_target}, {loc} 3.10 cutoff"[:31]
            df.to_excel(writer, sheet_name=sheet_name)
    
    print(f"\nExcel file saved to Downloads as '{excel_filename}'")
